# HW0: English Word Segmentation

## Task Description

Word segmentation is the task of splitting a continuous string of characters into individual words. In English, this is particularly challenging when spaces have been removed from text.

**Input:** A string with all spaces removed, e.g., `"choosespain"` or `"thisisatest"`

**Output:** A properly segmented sequence of words, e.g., `"choose spain"` or `"this is a test"`

The challenge is that there are often multiple valid ways to segment a string (e.g., `"choosespain"` could be `"choose spain"` or `"chooses pain"`), so we need a probabilistic model to select the most likely segmentation based on word frequencies in English text.

## Method Description

Both the baseline and improved solutions use a n-gram method to find the most probable word segmentation. The approach is based on [Peter Norvig's work in "Beautiful Data](https://norvig.com/ngrams/)."

### Core Algorithm

1. **Language Model:** Use word frequency counts from a [corpus](/hw0/data/count_1w.txt) to estimate P(word)
2. **Segmentation:** Use memoized recursion to find the segmentation with maximum probability
3. **Scoring:** Score a segmentation as the product (or sum of logs) of individual word probabilities

### Major Improvements (ensegment.py)

#### 1. Better Unknown Word Penalty

Baseline method assigns unknown words a uniform probability: `P(unknown) = 1/N`.  
However, in our method, unknown words are penalized based on their length to prefer shorter unknown words:

```python
#self.missingfn = missingfn or (lambda k, N: 1./N)
self.missingfn = missingfn or (lambda k, N: 1.0/(N * (10 ** len(k))))
```

This prevents the model from accepting very long, improbable words.  
We also tried different exponential base such as 5, 7, 10, 15, and found out 10 gave the best result.

#### 2. Smart Digit Handling

Digit Numbers should always be separated, e.g. `top 3 favourite comics` instead of `to p3 favourite comics`, and numbers should not be split (e.g., `10 people` should not become `1 0 people`). We modified the `segment()` function to:
- Detect leading digit sequences using regex
- Always treat the complete number as a single unit
- Assign a moderate probability (0.1) to digit sequences in the language model

```python 
    # Find leading digit sequence
    text = re.sub(r'(?<=\D)(?=\d)|(?<=\d)(?=\D)', ' ', text)
        if ' ' in text:
            out = []
            for t in text.split():
                if t.isdigit():
                    out.append(t)
                else:
                    out.extend(self.segment(t))
            return out
```

#### 3. Log Probability Scoring

Instead of multiplying probabilities (which can cause numerical underflow), we use the sum of log probabilities:

```python
# Baseline: product(self.Pw(w) for w in words)
# Improved: sum(math.log(self.Pw(w)) for w in words)
```

This is has no influence on the f1-score for out data, but using log should be more stable.

## Results

### Quantitative Results

We evaluated both methods on the dev and test data using F-score metrics. We also add an additional testing dataset (othertest.txt) to specifically evaluate the performance of handling digits.

| Method | dev.out | test.out| othertest.out |
|--------|---------|---------|---------------|
| Baseline (default.py) | 0.82  | 0.13 | 0.23
| Improvement 1 (avoid long unknowns) | 1.00 | 0.93 | 0.6
| Improvement 2 (smart handling digits) | 1.00 | 0.95 | 0.92
| Improvement 3 (log probability) | 1.00 | 0.95 | 0.92

### Qualitative Results

Here are some representative examples comparing baseline and improved segmentations:

#### Example 1: Many Rare Words (Unknown in the dictionary)
```
Input:    "itwasabrightcolddayinaprilandtheclockswerestrikingthirteen"
Baseline: "itwasabrightcoldda yinaprilandtheclocks werestrikingthirteen"     
Improved: "it was a bright cold day in april and the clocks were striking thirteen"      
```

#### Example 2: Digit Handling
```
Input:    "lovestoryin5words"
Baseline: "lovestoryin5words"     
Improved: "love story in 5 words"      
```

### Error Analysis

Some remaining errors in the improved model:

1. `10 people who mean alot to me`

Based on the data in `count_1w.txt`:
* the frequency for "alot" is 5929674
* the frequencies for "lot" is 106405208, for "a" is 9081174698
* N is 588124220187

by separating to "alot" the probability product will be 1.0082349606541626e-05, and by separating to "a" and "lot" the probability product will be 2.793616490258144e-06 which is smaller, that is why it is segmented as "alot"

2. `un regarded`

"unregarded" is an unknown word, which gives higher penalty

To solve the error, we need to word pairs `P(w2|w1)` instead of just individual words `P(w)` with the **bigram** method, however this requires additional dictionary to be referenced. This can be a future improvement direction.



In [2]:
alot = 5929674
a = 9081174698
lot = 106405208
N = 588124220187

prob_alot = alot / N
prob_a_lot = (a / N) * (lot / N)

print("P(alot):", prob_alot)
print("P(a lot):", prob_a_lot)


P(alot): 1.0082349606541626e-05
P(a lot): 2.793616490258144e-06
